In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark - Working with stringsa and dates")
    .master("local[*]")
    .getOrCreate()
)

spark

In [26]:
# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [27]:
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [28]:
emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [29]:
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [30]:
# M or F for Gender new column
from pyspark.sql.functions import when, col, expr

emp_gender_fixed = emp.withColumn("new_gender", when(col("gender") == 'Male', 'M')
                                  .when(col("gender") == 'Female', 'F')
                                  .otherwise(None)
                                 )

emp_gender_fixed_1 = emp.withColumn("new_gender", expr("case when gender = 'Male' then 'M' when gender='Female' then 'F' else null end"))

In [31]:
emp_gender_fixed.show()

+-----------+-------------+-------------+---+------+------+----------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|
+-----------+-------------+-------------+---+------+------+----------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|         F|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|         M|
|        010|   

In [32]:
# Replace in strings regexp
from pyspark.sql.functions import regexp_replace

emp_name_fixed = emp_gender_fixed.withColumn("new_name", regexp_replace(col("name"), "J", "Z"))

In [33]:
emp_name_fixed.show(5)

+-----------+-------------+----------+---+------+------+----------+----------+----------+
|employee_id|department_id|      name|age|gender|salary| hire_date|new_gender|  new_name|
+-----------+-------------+----------+---+------+------+----------+----------+----------+
|        001|          101|  John Doe| 30|  Male| 50000|2015-01-01|         M|  Zohn Doe|
|        002|          101|Jane Smith| 25|Female| 45000|2016-02-15|         F|Zane Smith|
|        003|          102| Bob Brown| 35|  Male| 55000|2014-05-01|         M| Bob Brown|
|        004|          102| Alice Lee| 28|Female| 48000|2017-09-30|         F| Alice Lee|
|        005|          103| Jack Chan| 40|  Male| 60000|2013-04-01|         M| Zack Chan|
+-----------+-------------+----------+---+------+------+----------+----------+----------+
only showing top 5 rows



In [36]:
# convert date
from pyspark.sql.functions import to_date
emp_date_fixed = emp_name_fixed.withColumn("hire_date", to_date(col("hire_date"), 'yyyy-MM-dd'))

In [37]:
emp_date_fixed.show(5)

+-----------+-------------+----------+---+------+------+----------+----------+----------+
|employee_id|department_id|      name|age|gender|salary| hire_date|new_gender|  new_name|
+-----------+-------------+----------+---+------+------+----------+----------+----------+
|        001|          101|  John Doe| 30|  Male| 50000|2015-01-01|         M|  Zohn Doe|
|        002|          101|Jane Smith| 25|Female| 45000|2016-02-15|         F|Zane Smith|
|        003|          102| Bob Brown| 35|  Male| 55000|2014-05-01|         M| Bob Brown|
|        004|          102| Alice Lee| 28|Female| 48000|2017-09-30|         F| Alice Lee|
|        005|          103| Jack Chan| 40|  Male| 60000|2013-04-01|         M| Zack Chan|
+-----------+-------------+----------+---+------+------+----------+----------+----------+
only showing top 5 rows



In [38]:
emp_date_fixed.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)



In [39]:
# add date columns
from pyspark.sql.functions import current_date, current_timestamp
emp_dated = emp_date_fixed.withColumn("date_now", current_date()).withColumn("timestamp_now", current_timestamp())

In [40]:
emp_dated.show(truncate=False)

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+----------+--------------------------+
|employee_id|department_id|name         |age|gender|salary|hire_date |new_gender|new_name     |date_now  |timestamp_now             |
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+----------+--------------------------+
|001        |101          |John Doe     |30 |Male  |50000 |2015-01-01|M         |Zohn Doe     |2026-05-17|2026-05-17 11:30:18.119866|
|002        |101          |Jane Smith   |25 |Female|45000 |2016-02-15|F         |Zane Smith   |2026-05-17|2026-05-17 11:30:18.119866|
|003        |102          |Bob Brown    |35 |Male  |55000 |2014-05-01|M         |Bob Brown    |2026-05-17|2026-05-17 11:30:18.119866|
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|F         |Alice Lee    |2026-05-17|2026-05-17 11:30:18.119866|
|005        |103          |Jack Chan    |40 |Male  |60000 |201

In [41]:
# Drop Null gender records

emp_1 = emp_dated.na.drop()

In [43]:
emp_1.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+----------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|  date_now|       timestamp_now|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+----------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|2026-05-17|2026-05-17 11:30:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|2026-05-17|2026-05-17 11:30:...|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|2026-05-17|2026-05-17 11:30:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|2026-05-17|2026-05-17 11:30:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|2026-05-1

In [46]:
# Transform Nulls to O or 0
from pyspark.sql.functions import coalesce, lit
emp_null_df = emp_dated.withColumn("new_gender", coalesce(col("new_gender"), lit("O")))

In [47]:
emp_null_df.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+----------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|  date_now|       timestamp_now|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+----------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|2026-05-17|2026-05-17 11:41:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|2026-05-17|2026-05-17 11:41:...|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|2026-05-17|2026-05-17 11:41:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|2026-05-17|2026-05-17 11:41:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|2026-05-1

In [49]:
# Drop old colums and fix new columns
emp_final = emp_null_df.drop("name", "gender").withColumnRenamed("new_name", "name").withColumnRenamed("new_gender", "gender")

In [50]:
emp_final.show()

+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+
|employee_id|department_id|age|salary| hire_date|gender|         name|  date_now|       timestamp_now|
+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+
|        001|          101| 30| 50000|2015-01-01|     M|     Zohn Doe|2026-05-17|2026-05-17 11:43:...|
|        002|          101| 25| 45000|2016-02-15|     F|   Zane Smith|2026-05-17|2026-05-17 11:43:...|
|        003|          102| 35| 55000|2014-05-01|     M|    Bob Brown|2026-05-17|2026-05-17 11:43:...|
|        004|          102| 28| 48000|2017-09-30|     F|    Alice Lee|2026-05-17|2026-05-17 11:43:...|
|        005|          103| 40| 60000|2013-04-01|     M|    Zack Chan|2026-05-17|2026-05-17 11:43:...|
|        006|          103| 32| 52000|2018-07-01|     F|    Zill Wong|2026-05-17|2026-05-17 11:43:...|
|        007|          101| 42| 70000|2012-03-15|     M|Zames Zohnson|202

In [51]:
# write into csv
emp_final.write.format("csv").save("data/output/4/emp.csv")

In [52]:
# Bonus - extract Day, Month, etc
# convert dates into strings and how to get parts
from pyspark.sql.functions import date_format

emp_fixed = emp_final.withColumn("date_string", date_format(col("hire_date"), "dd/MM/yyyy"))

In [53]:
emp_fixed.show()

+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+-----------+
|employee_id|department_id|age|salary| hire_date|gender|         name|  date_now|       timestamp_now|date_string|
+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+-----------+
|        001|          101| 30| 50000|2015-01-01|     M|     Zohn Doe|2026-05-17|2026-05-17 11:45:...| 01/01/2015|
|        002|          101| 25| 45000|2016-02-15|     F|   Zane Smith|2026-05-17|2026-05-17 11:45:...| 15/02/2016|
|        003|          102| 35| 55000|2014-05-01|     M|    Bob Brown|2026-05-17|2026-05-17 11:45:...| 01/05/2014|
|        004|          102| 28| 48000|2017-09-30|     F|    Alice Lee|2026-05-17|2026-05-17 11:45:...| 30/09/2017|
|        005|          103| 40| 60000|2013-04-01|     M|    Zack Chan|2026-05-17|2026-05-17 11:45:...| 01/04/2013|
|        006|          103| 32| 52000|2018-07-01|     F|    Zill Wong|2026-05-17

In [54]:
emp_fixed.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- gender: string (nullable = false)
 |-- name: string (nullable = true)
 |-- date_now: date (nullable = false)
 |-- timestamp_now: timestamp (nullable = false)
 |-- date_string: string (nullable = true)



In [55]:

emp_fixed = emp_final.withColumn("date_string", date_format(col("hire_date"), "yyyy"))

In [56]:
emp_fixed.show()

+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+-----------+
|employee_id|department_id|age|salary| hire_date|gender|         name|  date_now|       timestamp_now|date_string|
+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+-----------+
|        001|          101| 30| 50000|2015-01-01|     M|     Zohn Doe|2026-05-17|2026-05-17 11:47:...|       2015|
|        002|          101| 25| 45000|2016-02-15|     F|   Zane Smith|2026-05-17|2026-05-17 11:47:...|       2016|
|        003|          102| 35| 55000|2014-05-01|     M|    Bob Brown|2026-05-17|2026-05-17 11:47:...|       2014|
|        004|          102| 28| 48000|2017-09-30|     F|    Alice Lee|2026-05-17|2026-05-17 11:47:...|       2017|
|        005|          103| 40| 60000|2013-04-01|     M|    Zack Chan|2026-05-17|2026-05-17 11:47:...|       2013|
|        006|          103| 32| 52000|2018-07-01|     F|    Zill Wong|2026-05-17

In [57]:
emp_fixed = emp_final.withColumn("date_string", date_format(col("timestamp_now"), "z"))

In [58]:
emp_fixed.show(10)

+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+-----------+
|employee_id|department_id|age|salary| hire_date|gender|         name|  date_now|       timestamp_now|date_string|
+-----------+-------------+---+------+----------+------+-------------+----------+--------------------+-----------+
|        001|          101| 30| 50000|2015-01-01|     M|     Zohn Doe|2026-05-17|2026-05-17 11:48:...|        UTC|
|        002|          101| 25| 45000|2016-02-15|     F|   Zane Smith|2026-05-17|2026-05-17 11:48:...|        UTC|
|        003|          102| 35| 55000|2014-05-01|     M|    Bob Brown|2026-05-17|2026-05-17 11:48:...|        UTC|
|        004|          102| 28| 48000|2017-09-30|     F|    Alice Lee|2026-05-17|2026-05-17 11:48:...|        UTC|
|        005|          103| 40| 60000|2013-04-01|     M|    Zack Chan|2026-05-17|2026-05-17 11:48:...|        UTC|
|        006|          103| 32| 52000|2018-07-01|     F|    Zill Wong|2026-05-17